# Generate a Small Podcast from Notes

## Initializing setup

In [5]:
!pip install openai pydantic
!pip install datasets


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\dilia\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\dilia\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [6]:
!pip install openai pydub


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\dilia\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [7]:
!pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\dilia\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [8]:
import os
from openai import OpenAI 
from dotenv import load_dotenv 
from pydantic import BaseModel
from typing import List
load_dotenv() # load the .env file to get the environment variables

# Initialize client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
import sys
!{sys.executable} -m pip install python-dotenv 

## Process data from a source -pdf

In [11]:
# Step 1: Install the required library (run once)
!pip install pypdf


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\dilia\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [12]:
import sys
!{sys.executable} -m pip install pypdf

In [2]:
import os
print(os.getcwd()) #need to know the path to the pdf file to load it correctly

c:\Users\dilia\OneDrive\IronHack\Python_projects\Project1


In [13]:
# Step 2: Extract text from your PDF
from pypdf import PdfReader

# Load your PDF file
reader = PdfReader("c:\\Users\\dilia\\OneDrive\\IronHack\\Python_projects\\Project1\\Learning_AI_from_scratch.pdf")  # Actual file path

print(f"Total pages: {len(reader.pages)}")

# Extract all text
text = ""
for page in reader.pages:
    text += page.extract_text()

print(text[:2000])  # Preview the first 2000 characters

Total pages: 1
Stepping into the world of technology from a non-technical background can feel overwhelming, even 
intimidating at times. You might look at others who seem experienced and think, “Can I really do 
this?” Let me tell you something clearly—yes, you absolutely can. In fact, coming from a non-
technical background is not your weakness; it is your strength. You bring a unique perspective, real-
world understanding, and a different way of thinking that technology truly needs. You are not 
starting behind—you are starting differently, and that difference matters. 
This journey you’ve chosen is not meant to be easy. A short course may sound simple, but it 
demands time, focus, and consistent effort. There will be moments when nothing makes sense, 
when the code doesn’t work, when concepts feel too complex, and when you question your decision. 
But those moments are not signs to quit—they are signs that you are learning. Growth is 
uncomfortable. Learning something completely new

## Reprocess using LLM API

In [28]:
# ---- Step 1: Transform text into a podcast script ----

HOST_VOICES = {
    "Ankita": "nova",    # warm and friendly
    "Oli":    "shimmer", # soft and expressive
    "Alex":   "coral",   # clear and natural
    "Dilia":  "sage"     # calm and professional
}
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": """You are a podcast scriptwriter. 
            Transform the given text into an engaging, conversational podcast script 
            for four female hosts: Ankita, Oli, Alex and Dilia. 
            Each host has a distinct personality:
            - Ankita: warm and enthusiastic, loves analogies
            - Oli: thoughtful and asks great questions
            - Alex: clear and factual, provides examples
            - Dilia: calm summarizer, wraps up key points
            
            Use a friendly tone, add smooth transitions, 
            and make it flow naturally when spoken out loud.
            
            IMPORTANT: Format every line exactly like this:
            Ankita: Hello and welcome to the show!
            Oli: Great to be here today.
            
            Always start each line with the host name followed by a colon."""
        },
        {
            "role": "user",
            "content": f"Transform this into a podcast script:\n\n{text}"
        }
    ]
)

podcast_script = response.choices[0].message.content
print(podcast_script)

Ankita: Hello, listeners! Welcome back to another episode of our podcast. I'm Ankita, and today, we're diving into the exciting world of technology, especially for those of you coming from a non-technical background. Trust me, it's going to be an enlightening journey!

Oli: Hi everyone! It's Oli here. Now, I get it—stepping into tech without a technical background can seem daunting. Have you ever found yourself wondering, “Can I really do this?” 

Alex: Absolutely, Oli. And I want to jump in and say, yes, you can! You see, coming from a non-technical background actually gives you a unique advantage. It's not a setback but a different starting point with a fresh perspective that technology desperately needs.

Dilia: Hey there, it's Dilia. Remember, starting differently means your contributions will matter in distinct ways. Let’s explore why that’s so crucial and how you can make the most of it.

Ankita: Think of it like a garden. Every plant has a different journey to bloom, and tech is

## Generate audio file

In [30]:
import re
import os
from pydub import AudioSegment
from IPython.display import Audio, display

# ---- Voice assignment per host ----
HOST_VOICES = {
    "Ankita": "nova",    # warm and friendly
    "Oli":    "shimmer", # soft and expressive
    "Alex":   "coral",   # clear and natural
    "Dilia":  "sage"     # calm and professional
}

# ---- Step 2: Parse script into segments per speaker ----
def parse_script(script: str) -> list:
    segments = []
    pattern = r"^(Ankita|Oli|Alex|Dilia):\s*(.+)$"

    for line in script.splitlines():
        line = line.strip()
        match = re.match(pattern, line, re.IGNORECASE)
        if match:
            segments.append({
                "speaker": match.group(1).capitalize(),
                "line":    match.group(2).strip()
            })

    if not segments:
        raise ValueError(
            "Could not parse any speaker lines. "
            "Make sure the script uses 'SpeakerName: text' format."
        )

    print(f"✅ Parsed {len(segments)} segments across "
          f"{len(set(s['speaker'] for s in segments))} speakers")
    return segments


# ---- Step 3: Generate one audio file per segment ----
def generate_audio_segments(segments: list, output_dir: str = "audio_segments") -> list:
    os.makedirs(output_dir, exist_ok=True)
    audio_files = []

    for i, segment in enumerate(segments):
        speaker = segment["speaker"]
        line    = segment["line"]
        voice   = HOST_VOICES.get(speaker, "nova")  # fallback to nova
        path    = os.path.join(output_dir, f"segment_{i:03d}_{speaker}.mp3")

        print(f"🎙️ [{i+1}/{len(segments)}] {speaker} ({voice}): {line[:60]}...")

        response = client.audio.speech.create(
            model="tts-1-hd",
            voice=voice,
            input=line
        )
        response.stream_to_file(path)
        audio_files.append(path)

    print(f"✅ Generated {len(audio_files)} audio segments")
    return audio_files


# ---- Step 4: Combine all segments into one MP3 ----
def combine_audio(audio_files: list, output_path: str = "podcast_episode.mp3") -> str:
    print("🔗 Combining audio segments...")
    combined = AudioSegment.empty()

    for path in audio_files:
        segment = AudioSegment.from_mp3(path)
        combined += segment + AudioSegment.silent(duration=300)  # 300ms pause between speakers

    combined.export(output_path, format="mp3")
    print(f"✅ Final podcast saved to: {output_path}")
    return output_path


# ---- Run the pipeline ----
segments    = parse_script(podcast_script)
audio_files = generate_audio_segments(segments)
output_path = combine_audio(audio_files)

# ---- Listen to the final podcast ----
display(Audio(output_path))

✅ Parsed 16 segments across 4 speakers
🎙️ [1/16] Ankita (nova): Hello, listeners! Welcome back to another episode of our pod...


C:\Users\dilia\AppData\Local\Temp\ipykernel_7316\435966132.py:57: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(path)


🎙️ [2/16] Oli (shimmer): Hi everyone! It's Oli here. Now, I get it—stepping into tech...
🎙️ [3/16] Alex (coral): Absolutely, Oli. And I want to jump in and say, yes, you can...
🎙️ [4/16] Dilia (sage): Hey there, it's Dilia. Remember, starting differently means ...
🎙️ [5/16] Ankita (nova): Think of it like a garden. Every plant has a different journ...
🎙️ [6/16] Oli (shimmer): That’s a beautiful analogy, Ankita. So, how can someone with...
🎙️ [7/16] Alex (coral): Great question, Oli! It's all about taking things one small ...
🎙️ [8/16] Dilia (sage): Right, Alex. Those moments of confusion or complexity? They’...
🎙️ [9/16] Ankita (nova): Imagine climbing a mountain. Every step you take, no matter ...
🎙️ [10/16] Oli (shimmer): Exactly, Ankita. But what about those days when motivation j...
🎙️ [11/16] Alex (coral): Discipline is your ally on those days, Oli. Even a small eff...
🎙️ [12/16] Dilia (sage): Over time, those small efforts lead to big changes. And you'...
🎙️ [13/16] Ankita (nova)

## Use Python data structures

In [32]:
import os
import re
from dataclasses import dataclass, field
from openai import OpenAI
from dotenv import load_dotenv
from pypdf import PdfReader
from pydub import AudioSegment
from IPython.display import Audio, display

# ---- Configuration ----
@dataclass
class PodcastConfig:
    model_chat: str = "gpt-4o"
    model_tts: str = "tts-1-hd"
    model_whisper: str = "whisper-1"
    language: str = "en"
    output_file: str = "podcast_episode.mp3"
    audio_segments_dir: str = "audio_segments"
    host_voices: dict = field(default_factory=lambda: {
        "Ankita": "nova",    # warm and friendly
        "Oli":    "shimmer", # soft and expressive
        "Alex":   "coral",   # clear and natural
        "Dilia":  "sage"     # calm and professional
    })


# ---- Data Structures ----
@dataclass
class PDFContent:
    file_path: str
    total_pages: int = 0
    raw_text: str = ""

    def load(self):
        reader = PdfReader(self.file_path)
        self.total_pages = len(reader.pages)
        self.raw_text = "".join(page.extract_text() for page in reader.pages)
        print(f"✅ Loaded PDF: {self.total_pages} pages, {len(self.raw_text)} characters")


@dataclass
class PodcastEpisode:
    title: str
    source_text: str
    script: str = ""
    audio_path: str = ""
    transcription: str = ""
    metadata: dict = field(default_factory=dict)


# ---- Processor Class ----
class PodcastProcessor:
    def __init__(self, config: PodcastConfig):
        load_dotenv()
        self.config = config
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    def generate_script(self, episode: PodcastEpisode) -> PodcastEpisode:
        print("🎙️ Generating podcast script...")
        host_names = ", ".join(self.config.host_voices.keys())
        response = self.client.chat.completions.create(
            model=self.config.model_chat,
            messages=[
                {
                    "role": "system",
                    "content": f"""You are a podcast scriptwriter.
                    Transform the given text into an engaging, conversational podcast script
                    for four female hosts: {host_names}.
                    Each host has a distinct personality:
                    - Ankita: warm and enthusiastic, loves analogies
                    - Oli: thoughtful and asks great questions
                    - Alex: clear and factual, provides examples
                    - Dilia: calm summarizer, wraps up key points

                    Use a friendly tone, add smooth transitions,
                    and make it flow naturally when spoken out loud.
                    Keep it between 3-5 minutes when read aloud.

                    IMPORTANT: Format every line exactly like this:
                    Ankita: Hello and welcome to the show!
                    Oli: Great to be here today.

                    Always start each line with the host name followed by a colon."""
                },
                {
                    "role": "user",
                    "content": f"Transform this into a podcast script:\n\n{episode.source_text}"
                }
            ]
        )
        episode.script = response.choices[0].message.content
        episode.metadata["script_length"] = len(episode.script)
        print(f"✅ Script generated: {len(episode.script)} characters")
        return episode

    def parse_script(self, script: str) -> list:
        """Parse script into list of {speaker, line} dicts."""
        segments = []
        hosts = "|".join(self.config.host_voices.keys())
        pattern = rf"^({hosts}):\s*(.+)$"

        for line in script.splitlines():
            line = line.strip()
            match = re.match(pattern, line, re.IGNORECASE)
            if match:
                segments.append({
                    "speaker": match.group(1).capitalize(),
                    "line":    match.group(2).strip()
                })

        if not segments:
            raise ValueError(
                "Could not parse any speaker lines. "
                "Make sure the script uses 'SpeakerName: text' format."
            )

        speakers_found = set(s['speaker'] for s in segments)
        print(f"✅ Parsed {len(segments)} segments across {len(speakers_found)} speakers: {speakers_found}")
        return segments

    def generate_audio(self, episode: PodcastEpisode) -> PodcastEpisode:
        print("🔊 Generating audio per host...")
        os.makedirs(self.config.audio_segments_dir, exist_ok=True)

        segments   = self.parse_script(episode.script)
        audio_files = []

        for i, segment in enumerate(segments):
            speaker = segment["speaker"]
            line    = segment["line"]
            voice   = self.config.host_voices.get(speaker, "nova")
            path    = os.path.join(
                self.config.audio_segments_dir,
                f"segment_{i:03d}_{speaker}.mp3"
            )

            print(f"🎙️ [{i+1}/{len(segments)}] {speaker} ({voice}): {line[:60]}...")

            response = self.client.audio.speech.create(
                model=self.config.model_tts,
                voice=voice,
                input=line
            )
            response.stream_to_file(path)
            audio_files.append(path)

        # Combine all segments into one MP3
        print("🔗 Combining audio segments...")
        combined = AudioSegment.empty()
        for path in audio_files:
            combined += AudioSegment.from_mp3(path) + AudioSegment.silent(duration=300)

        episode.audio_path = self.config.output_file
        combined.export(episode.audio_path, format="mp3")
        episode.metadata["audio_file"]    = episode.audio_path
        episode.metadata["total_segments"] = len(segments)
        print(f"✅ Audio saved: {episode.audio_path}")
        return episode

    def transcribe_audio(self, episode: PodcastEpisode) -> PodcastEpisode:
        print("📝 Transcribing audio with Whisper...")
        with open(episode.audio_path, "rb") as audio_file:
            transcription = self.client.audio.transcriptions.create(
                model=self.config.model_whisper,
                file=audio_file,
                language=self.config.language
            )
        episode.transcription = transcription.text
        print(f"✅ Transcription complete: {len(episode.transcription)} characters")
        return episode

    def play_audio(self, episode: PodcastEpisode):
        print("▶️ Playing podcast...")
        display(Audio(episode.audio_path))

    def run(self, pdf_path: str, title: str) -> PodcastEpisode:
        # 1. Load PDF
        pdf = PDFContent(file_path=pdf_path)
        pdf.load()

        # 2. Create episode
        episode = PodcastEpisode(title=title, source_text=pdf.raw_text)

        # 3. Run pipeline
        episode = self.generate_script(episode)
        episode = self.generate_audio(episode)
        episode = self.transcribe_audio(episode)
        self.play_audio(episode)

        return episode


# ---- Run it ----
config    = PodcastConfig()
processor = PodcastProcessor(config)

episode = processor.run(
    pdf_path=r"c:\\Users\\dilia\\OneDrive\\IronHack\\Python_projects\\Project1\\Learning_AI_from_scratch.pdf",
    title="Learning AI from Scratch"
)

# ---- Summary ----
print("\n📋 Episode Summary:")
print(f"  Title:          {episode.title}")
print(f"  Script length:  {episode.metadata['script_length']} characters")
print(f"  Total segments: {episode.metadata['total_segments']}")
print(f"  Audio file:     {episode.metadata['audio_file']}")
print(f"  Transcription:  {episode.transcription[:200]}...")

✅ Loaded PDF: 1 pages, 2444 characters
🎙️ Generating podcast script...
✅ Script generated: 3034 characters
🔊 Generating audio per host...
✅ Parsed 16 segments across 4 speakers: {'Dilia', 'Alex', 'Ankita', 'Oli'}
🎙️ [1/16] Ankita (nova): Hey everyone! Welcome back to our podcast, where today we're...


C:\Users\dilia\AppData\Local\Temp\ipykernel_7316\1918503081.py:144: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(path)


🎙️ [2/16] Oli (shimmer): Oh, that can definitely feel overwhelming, right? Especially...
🎙️ [3/16] Alex (coral): That’s a common feeling, Oli! But let’s set the record strai...
🎙️ [4/16] Dilia (sage): Exactly, Alex. You’re bringing that fresh, unique perspectiv...
🎙️ [5/16] Ankita (nova): And it’s not like the path is meant to be easy, right? Think...
🎙️ [6/16] Oli (shimmer): Oh, I've been there when nothing seems to make sense. What d...
🎙️ [7/16] Alex (coral): Those are signs, Oli, that you're learning! Remember, growth...
🎙️ [8/16] Dilia (sage): Yes, one step today, another tomorrow—transformation happens...
🎙️ [9/16] Ankita (nova): There’ll be those days when you're energized, and others whe...
🎙️ [10/16] Oli (shimmer): I feel you, Ankita. But on those tough days, it’s about disc...
🎙️ [11/16] Alex (coral): Over time, these small, steady efforts build up into somethi...
🎙️ [12/16] Dilia (sage): Right, you’re proving to yourself that you can adapt and suc...
🎙️ [13/16] Ankita (nova)


📋 Episode Summary:
  Title:          Learning AI from Scratch
  Script length:  3034 characters
  Total segments: 16
  Audio file:     podcast_episode.mp3
  Transcription:  Hey everyone, welcome back to our podcast where today we're diving into a topic that might feel like trying to learn how to ride a unicycle while juggling. We're talking about stepping into the world ...


## Build Gradio interface

In [18]:
!pip install gradio

  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   -- ------------------------------------- 1.3/19.7 MB 7.7 MB/s eta 0:00:03
   ----- ---------------------------------- 2.6/19.7 MB 7.5 MB/s eta 0:00:03
   -------- ------------------------------- 4.2/19.7 MB 6.9 MB/s eta 0:00:03
   ----------- ---------------------------- 5.8/19.7 MB 7.1 MB/s eta 0:00:02
   -------------- ------------------------- 7.3/19.7 MB 7.1 MB/s eta 0:00:02
   ------------------ --------------------- 8.9/19.7 MB 7.1 MB/s eta 0:00:02
   --------------------- ------------------ 10.5/19.7 MB 7.1 MB/s eta 0:00:02
   ------------------------ --------------- 12.1/19.7 MB 7.2 MB/s eta 0:00:02
   --------------------------- ------------ 13.4/19.7 MB 7.2 MB/s eta 0:00:01
   ------------------------------ --------- 14.9/19.7 MB 7.2 MB/s eta 0:00:01
   --------------------------------- ------ 16.5/19.7 MB 7.2 MB/s eta 0:00:01
  

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\dilia\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [21]:
import sys
!{sys.executable} -m pip install gradio

  Using cached gradio-6.13.0-py3-none-any.whl.metadata (17 kB)
  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
  Using cached brotli-1.2.0-cp314-cp314-win_amd64.whl.metadata (6.3 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached huggingface_hub-1.12.0-py3-none-any.whl.metadata (14 kB)
  Using cached orjson-3.11.8-cp314-cp314-win_amd64.whl.metadata (43 kB)
  Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached python_multipart-0.0.27-py3-none-any.whl.metadata (2.1 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using c

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [33]:
import os
import re
import gradio as gr
from dataclasses import dataclass, field
from openai import OpenAI
from dotenv import load_dotenv
from pypdf import PdfReader
from pydub import AudioSegment

# ---- Configuration ----
@dataclass
class PodcastConfig:
    model_chat: str = "gpt-4o"
    model_tts: str = "tts-1-hd"
    model_whisper: str = "whisper-1"
    language: str = "en"
    output_file: str = "podcast_episode.mp3"
    audio_segments_dir: str = "audio_segments"
    host_voices: dict = field(default_factory=lambda: {
        "Ankita": "nova",
        "Oli":    "shimmer",
        "Alex":   "coral",
        "Dilia":  "sage"
    })


# ---- Data Structures ----
@dataclass
class PDFContent:
    file_path: str
    total_pages: int = 0
    raw_text: str = ""

    def load(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"PDF file not found: {self.file_path}")
        if not self.file_path.lower().endswith(".pdf"):
            raise ValueError("Uploaded file is not a PDF.")
        reader = PdfReader(self.file_path)
        if len(reader.pages) == 0:
            raise ValueError("The PDF has no pages.")
        self.total_pages = len(reader.pages)
        self.raw_text = "".join(page.extract_text() or "" for page in reader.pages)
        if not self.raw_text.strip():
            raise ValueError("No text could be extracted. The PDF may be scanned/image-based.")


@dataclass
class PodcastEpisode:
    title: str
    source_text: str
    script: str = ""
    audio_path: str = ""
    transcription: str = ""
    metadata: dict = field(default_factory=dict)


# ---- Processor ----
class PodcastProcessor:
    def __init__(self, config: PodcastConfig):
        load_dotenv()
        self.config = config
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    def generate_script(self, episode: PodcastEpisode) -> PodcastEpisode:
        host_names = ", ".join(self.config.host_voices.keys())
        response = self.client.chat.completions.create(
            model=self.config.model_chat,
            messages=[
                {
                    "role": "system",
                    "content": f"""You are a podcast scriptwriter.
                    Transform the given text into an engaging, conversational podcast script
                    for four female hosts: {host_names}.
                    Each host has a distinct personality:
                    - Ankita: warm and enthusiastic, loves analogies
                    - Oli: thoughtful and asks great questions
                    - Alex: clear and factual, provides examples
                    - Dilia: calm summarizer, wraps up key points

                    Use a friendly tone, add smooth transitions,
                    and make it flow naturally when spoken out loud.
                    Keep it between 3-5 minutes when read aloud.

                    IMPORTANT: Format every line exactly like this:
                    Ankita: Hello and welcome to the show!
                    Oli: Great to be here today.

                    Always start each line with the host name followed by a colon."""
                },
                {
                    "role": "user",
                    "content": f"Transform this into a podcast script:\n\n{episode.source_text}"
                }
            ]
        )
        episode.script = response.choices[0].message.content
        episode.metadata["script_length"] = len(episode.script)
        return episode

    def parse_script(self, script: str) -> list:
        segments = []
        hosts = "|".join(self.config.host_voices.keys())
        pattern = rf"^({hosts}):\s*(.+)$"
        for line in script.splitlines():
            line = line.strip()
            match = re.match(pattern, line, re.IGNORECASE)
            if match:
                segments.append({
                    "speaker": match.group(1).capitalize(),
                    "line":    match.group(2).strip()
                })
        if not segments:
            raise ValueError(
                "Could not parse any speaker lines. "
                "Make sure the script uses 'SpeakerName: text' format."
            )
        return segments

    def generate_audio(self, episode: PodcastEpisode) -> PodcastEpisode:
        os.makedirs(self.config.audio_segments_dir, exist_ok=True)
        segments = self.parse_script(episode.script)
        audio_files = []

        for i, segment in enumerate(segments):
            speaker = segment["speaker"]
            line    = segment["line"]
            voice   = self.config.host_voices.get(speaker, "nova")
            path    = os.path.join(
                self.config.audio_segments_dir,
                f"segment_{i:03d}_{speaker}.mp3"
            )
            response = self.client.audio.speech.create(
                model=self.config.model_tts,
                voice=voice,
                input=line
            )
            response.stream_to_file(path)
            audio_files.append(path)

        # Combine all segments
        combined = AudioSegment.empty()
        for path in audio_files:
            combined += AudioSegment.from_mp3(path) + AudioSegment.silent(duration=300)

        episode.audio_path = self.config.output_file
        combined.export(episode.audio_path, format="mp3")
        episode.metadata["audio_file"]     = episode.audio_path
        episode.metadata["total_segments"] = len(segments)
        return episode

    def transcribe_audio(self, episode: PodcastEpisode) -> PodcastEpisode:
        with open(episode.audio_path, "rb") as audio_file:
            transcription = self.client.audio.transcriptions.create(
                model=self.config.model_whisper,
                file=audio_file,
                language=self.config.language
            )
        episode.transcription = transcription.text
        return episode


# ---- Gradio Handler ----
def generate_podcast(pdf_file, title, tts_model):
    try:
        if pdf_file is None:
            yield "❌ Please upload a PDF file.", None, "", ""
            return
        if not title or not title.strip():
            yield "❌ Please enter an episode title.", None, "", ""
            return

        config = PodcastConfig(model_tts=tts_model)
        processor = PodcastProcessor(config)

        # Load PDF
        yield "📄 Loading PDF...", None, "", ""
        pdf = PDFContent(file_path=pdf_file.name)
        pdf.load()

        # Create episode
        episode = PodcastEpisode(title=title.strip(), source_text=pdf.raw_text)

        # Generate script
        yield "✍️ Generating 4-host podcast script...", None, "", ""
        episode = processor.generate_script(episode)

        # Generate audio
        yield "🔊 Generating audio for each host (this may take a while)...", None, episode.script, ""
        episode = processor.generate_audio(episode)

        # Transcribe
        yield "📝 Transcribing with Whisper...", None, episode.script, ""
        episode = processor.transcribe_audio(episode)

        # Summary
        summary = f"""✅ Done!

📌 Title:           {episode.title}
📄 Pages:           {pdf.total_pages}
✍️  Script length:   {episode.metadata['script_length']} characters
🎙️ Hosts:           Ankita (nova), Oli (shimmer), Alex (coral), Dilia (sage)
🔊 Segments:        {episode.metadata['total_segments']}
🤖 TTS Model:       {tts_model}"""

        yield summary, episode.audio_path, episode.script, episode.transcription

    except Exception as e:
        yield f"❌ Error: {str(e)}", None, "", ""


# ---- Gradio UI ----
with gr.Blocks(title="🎙️ PDF to Podcast", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🎙️ PDF to Podcast Generator")
    gr.Markdown("Upload a PDF and turn it into a podcast with **4 female hosts**: Ankita, Oli, Alex & Dilia.")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="📄 Upload PDF",
                file_types=[".pdf"]
            )
            title_input = gr.Textbox(
                label="🎙️ Episode Title",
                placeholder="e.g. Learning AI from Scratch",
                value="My Podcast Episode"
            )
            model_input = gr.Dropdown(
                label="🤖 TTS Model",
                choices=["tts-1", "tts-1-hd"],
                value="tts-1-hd"
            )
            gr.Markdown("""**🎙️ Host Voices:**  
            Ankita → nova | Oli → shimmer  
            Alex → coral | Dilia → sage""")
            generate_btn = gr.Button("🚀 Generate Podcast", variant="primary")

        with gr.Column(scale=2):
            status_output = gr.Textbox(label="📊 Status", lines=10)
            audio_output = gr.Audio(label="▶️ Listen to Podcast", type="filepath")

    with gr.Row():
        with gr.Tabs():
            with gr.Tab("📝 Podcast Script"):
                script_output = gr.Textbox(label="Generated Script", lines=15)
            with gr.Tab("🔤 Whisper Transcription"):
                transcription_output = gr.Textbox(label="Transcription", lines=15)

    generate_btn.click(
        fn=generate_podcast,
        inputs=[pdf_input, title_input, model_input],
        outputs=[status_output, audio_output, script_output, transcription_output]
    )

demo.launch(share=False)

C:\Users\dilia\AppData\Local\Temp\ipykernel_7316\3896878160.py:213: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="🎙️ PDF to Podcast", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


C:\Users\dilia\AppData\Local\Temp\ipykernel_7316\3896878160.py:138: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(path)


## Adding error handlig

In [ ]:
import os
import re
import gradio as gr
from dataclasses import dataclass, field
from openai import OpenAI, APIConnectionError, AuthenticationError, RateLimitError, APIStatusError
from dotenv import load_dotenv
from pypdf import PdfReader
from pydub import AudioSegment

# ---- Configuration ----
@dataclass
class PodcastConfig:
    model_chat: str = "gpt-4o"
    model_tts: str = "tts-1-hd"
    model_whisper: str = "whisper-1"
    language: str = "en"
    output_file: str = "podcast_episode.mp3"
    audio_segments_dir: str = "audio_segments"
    host_voices: dict = field(default_factory=lambda: {
        "Ankita": "nova",    # warm and friendly
        "Oli":    "shimmer", # soft and expressive
        "Alex":   "coral",   # clear and natural
        "Dilia":  "sage"     # calm and professional
    })


# ---- Data Structures ----
@dataclass
class PDFContent:
    file_path: str
    total_pages: int = 0
    raw_text: str = ""

    def load(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"PDF file not found: {self.file_path}")
        if not self.file_path.lower().endswith(".pdf"):
            raise ValueError("Uploaded file is not a PDF.")
        reader = PdfReader(self.file_path)
        if len(reader.pages) == 0:
            raise ValueError("The PDF has no pages.")
        self.total_pages = len(reader.pages)
        self.raw_text = "".join(page.extract_text() or "" for page in reader.pages)
        if not self.raw_text.strip():
            raise ValueError(
                "No text could be extracted from this PDF. "
                "It may be a scanned/image-based PDF. "
                "Please use a text-based PDF."
            )


@dataclass
class PodcastEpisode:
    title: str
    source_text: str
    script: str = ""
    audio_path: str = ""
    transcription: str = ""
    metadata: dict = field(default_factory=dict)


# ---- Custom Exceptions ----
class PodcastGenerationError(Exception):
    pass

class PDFLoadError(PodcastGenerationError):
    pass

class ScriptGenerationError(PodcastGenerationError):
    pass

class AudioGenerationError(PodcastGenerationError):
    pass

class TranscriptionError(PodcastGenerationError):
    pass


# ---- Processor ----
class PodcastProcessor:
    def __init__(self, config: PodcastConfig):
        load_dotenv()
        api_key = os.getenv("OPENAI_API_KEY")

        if not api_key:
            raise EnvironmentError(
                "OPENAI_API_KEY is missing. "
                "Please add it to your .env file."
            )
        if not api_key.startswith("sk-"):
            raise EnvironmentError(
                "OPENAI_API_KEY looks invalid. "
                "It should start with 'sk-'."
            )

        self.config = config
        self.client = OpenAI(api_key=api_key)

    def generate_script(self, episode: PodcastEpisode) -> PodcastEpisode:
        try:
            host_names = ", ".join(self.config.host_voices.keys())
            response = self.client.chat.completions.create(
                model=self.config.model_chat,
                messages=[
                    {
                        "role": "system",
                        "content": f"""You are a podcast scriptwriter.
                        Transform the given text into an engaging, conversational podcast script
                        for four female hosts: {host_names}.
                        Each host has a distinct personality:
                        - Ankita: warm and enthusiastic, loves analogies
                        - Oli: thoughtful and asks great questions
                        - Alex: clear and factual, provides examples
                        - Dilia: calm summarizer, wraps up key points

                        Use a friendly tone, add smooth transitions,
                        and make it flow naturally when spoken out loud.
                        Keep it between 3-5 minutes when read aloud.

                        IMPORTANT: Format every line exactly like this:
                        Ankita: Hello and welcome to the show!
                        Oli: Great to be here today.

                        Always start each line with the host name followed by a colon."""
                    },
                    {
                        "role": "user",
                        "content": f"Transform this into a podcast script:\n\n{episode.source_text}"
                    }
                ]
            )
            episode.script = response.choices[0].message.content

            if not episode.script or not episode.script.strip():
                raise ScriptGenerationError("OpenAI returned an empty script.")

            episode.metadata["script_length"] = len(episode.script)
            return episode

        except AuthenticationError:
            raise ScriptGenerationError("Invalid OpenAI API key. Please check your .env file.")
        except RateLimitError:
            raise ScriptGenerationError("OpenAI rate limit reached. Please wait a moment and try again.")
        except APIConnectionError:
            raise ScriptGenerationError("Could not connect to OpenAI. Please check your internet connection.")
        except APIStatusError as e:
            raise ScriptGenerationError(f"OpenAI API error: {e.status_code} - {e.message}")

    def parse_script(self, script: str) -> list:
        segments = []
        hosts = "|".join(self.config.host_voices.keys())
        pattern = rf"^({hosts}):\s*(.+)$"

        for line in script.splitlines():
            line = line.strip()
            match = re.match(pattern, line, re.IGNORECASE)
            if match:
                segments.append({
                    "speaker": match.group(1).capitalize(),
                    "line":    match.group(2).strip()
                })

        if not segments:
            raise AudioGenerationError(
                "Could not parse any speaker lines from the script. "
                "Please try generating again."
            )
        return segments

    def generate_audio(self, episode: PodcastEpisode) -> PodcastEpisode:
        try:
            if not episode.script:
                raise AudioGenerationError("No script available to convert to audio.")

            os.makedirs(self.config.audio_segments_dir, exist_ok=True)
            segments = self.parse_script(episode.script)
            audio_files = []

            for i, segment in enumerate(segments):
                speaker = segment["speaker"]
                line    = segment["line"]
                voice   = self.config.host_voices.get(speaker, "nova")
                path    = os.path.join(
                    self.config.audio_segments_dir,
                    f"segment_{i:03d}_{speaker}.mp3"
                )

                response = self.client.audio.speech.create(
                    model=self.config.model_tts,
                    voice=voice,
                    input=line
                )
                response.stream_to_file(path)
                audio_files.append(path)

            # Combine all segments
            combined = AudioSegment.empty()
            for path in audio_files:
                combined += AudioSegment.from_mp3(path) + AudioSegment.silent(duration=300)

            episode.audio_path = self.config.output_file
            combined.export(episode.audio_path, format="mp3")
            episode.metadata["audio_file"]     = episode.audio_path
            episode.metadata["total_segments"] = len(segments)

            if not os.path.exists(episode.audio_path):
                raise AudioGenerationError("Audio file was not created successfully.")

            return episode

        except AuthenticationError:
            raise AudioGenerationError("Invalid OpenAI API key during audio generation.")
        except RateLimitError:
            raise AudioGenerationError("OpenAI rate limit reached during audio generation. Please try again shortly.")
        except APIConnectionError:
            raise AudioGenerationError("Connection lost during audio generation. Please check your internet.")
        except APIStatusError as e:
            raise AudioGenerationError(f"Audio API error: {e.status_code} - {e.message}")

    def transcribe_audio(self, episode: PodcastEpisode) -> PodcastEpisode:
        try:
            if not os.path.exists(episode.audio_path):
                raise TranscriptionError(
                    f"Audio file not found for transcription: {episode.audio_path}"
                )

            with open(episode.audio_path, "rb") as audio_file:
                transcription = self.client.audio.transcriptions.create(
                    model=self.config.model_whisper,
                    file=audio_file,
                    language=self.config.language
                )

            episode.transcription = transcription.text

            if not episode.transcription.strip():
                episode.transcription = "⚠️ Transcription returned empty. The audio may be silent or too short."

            return episode

        except AuthenticationError:
            raise TranscriptionError("Invalid OpenAI API key during transcription.")
        except RateLimitError:
            raise TranscriptionError("Rate limit reached during transcription. Please try again shortly.")
        except APIConnectionError:
            raise TranscriptionError("Connection lost during transcription. Please check your internet.")
        except APIStatusError as e:
            raise TranscriptionError(f"Transcription API error: {e.status_code} - {e.message}")


# ---- Gradio Handler ----
def generate_podcast(pdf_file, title, tts_model):

    if pdf_file is None:
        yield "❌ Please upload a PDF file.", None, "", ""
        return

    if not title or not title.strip():
        yield "❌ Please enter an episode title.", None, "", ""
        return

    try:
        config = PodcastConfig(model_tts=tts_model)

        try:
            processor = PodcastProcessor(config)
        except EnvironmentError as e:
            yield f"❌ Configuration Error:\n{str(e)}", None, "", ""
            return

        # Load PDF
        yield "📄 Loading PDF...", None, "", ""
        try:
            pdf = PDFContent(file_path=pdf_file.name)
            pdf.load()
        except (FileNotFoundError, ValueError) as e:
            yield f"❌ PDF Error:\n{str(e)}", None, "", ""
            return

        episode = PodcastEpisode(title=title.strip(), source_text=pdf.raw_text)

        # Generate script
        yield "✍️ Generating 4-host podcast script...", None, "", ""
        try:
            episode = processor.generate_script(episode)
        except ScriptGenerationError as e:
            yield f"❌ Script Generation Failed:\n{str(e)}", None, "", ""
            return

        # Generate audio
        yield "🔊 Generating audio for each host (this may take a while)...", None, episode.script, ""
        try:
            episode = processor.generate_audio(episode)
        except AudioGenerationError as e:
            yield f"❌ Audio Generation Failed:\n{str(e)}", None, episode.script, ""
            return

        # Transcribe
        yield "📝 Transcribing with Whisper...", None, episode.script, ""
        try:
            episode = processor.transcribe_audio(episode)
        except TranscriptionError as e:
            yield (
                f"⚠️ Transcription failed (audio is still available):\n{str(e)}",
                episode.audio_path,
                episode.script,
                "Transcription unavailable."
            )
            return

        # Summary
        summary = f"""✅ Podcast generated successfully!

📌 Title:           {episode.title}
📄 Pages:           {pdf.total_pages}
✍️  Script length:   {episode.metadata['script_length']} characters
🎙️ Hosts:           Ankita (nova), Oli (shimmer), Alex (coral), Dilia (sage)
🔊 Segments:        {episode.metadata['total_segments']}
🤖 TTS Model:       {tts_model}"""

        yield summary, episode.audio_path, episode.script, episode.transcription

    except Exception as e:
        yield f"❌ Unexpected error:\n{str(e)}\n\nPlease try again or contact support.", None, "", ""


# ---- Gradio UI ----
with gr.Blocks(title="🎙️ PDF to Podcast", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🎙️ PDF to Podcast Generator")
    gr.Markdown("Upload a PDF and turn it into a podcast with **4 female hosts**: Ankita, Oli, Alex & Dilia.")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="📄 Upload PDF",
                file_types=[".pdf"]
            )
            title_input = gr.Textbox(
                label="🎙️ Episode Title",
                placeholder="e.g. Learning AI from Scratch",
                value="My Podcast Episode"
            )
            model_input = gr.Dropdown(
                label="🤖 TTS Model",
                choices=["tts-1", "tts-1-hd"],
                value="tts-1-hd"
            )
            gr.Markdown("""**🎙️ Host Voices:**  
            Ankita → nova | Oli → shimmer  
            Alex → coral | Dilia → sage""")
            generate_btn = gr.Button("🚀 Generate Podcast", variant="primary")

        with gr.Column(scale=2):
            status_output = gr.Textbox(label="📊 Status", lines=10)
            audio_output = gr.Audio(label="▶️ Listen to Podcast", type="filepath")

    with gr.Row():
        with gr.Tabs():
            with gr.Tab("📝 Podcast Script"):
                script_output = gr.Textbox(label="Generated Script", lines=15)
            with gr.Tab("🔤 Whisper Transcription"):
                transcription_output = gr.Textbox(label="Transcription", lines=15)

    generate_btn.click(
        fn=generate_podcast,
        inputs=[pdf_input, title_input, model_input],
        outputs=[status_output, audio_output, script_output, transcription_output]
    )

demo.launch(share=True)

C:\Users\dilia\AppData\Local\Temp\ipykernel_7316\1880504014.py:328: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="🎙️ PDF to Podcast", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://14122211075fc67b4d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


C:\Users\dilia\AppData\Local\Temp\ipykernel_7316\1880504014.py:193: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(path)
C:\Users\dilia\AppData\Local\Temp\ipykernel_7316\1880504014.py:193: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(path)
